In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# PATHS (COLAB GPU)
# ---------------------------------------------------------
DATASET_PATH = "/content/drive/MyDrive/Datasets/mango/train"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Datasets/mango/model_mango_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 8


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model
# ---------------------------------------------------------
def build_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("GPU Available:", tf.config.list_physical_devices('GPU'))

    print("Loading dataset...")
    df = load_dataset(DATASET_PATH)

    train_df, val_df = train_test_split(
        df,
        train_size=0.8,
        stratify=df["Label"],
        random_state=42
    )

    # 🔥 STRONG AUGMENTATION (important for small dataset)
    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=30,
        zoom_range=0.3,
        width_shift_range=0.2,
        height_shift_range=0.2,
        brightness_range=[0.6, 1.4],
        shear_range=0.2,
        horizontal_flip=True,
        fill_mode="nearest"
    )

    val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = val_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Classes:", train_images.class_indices)

    model = build_model(len(train_images.class_indices))

    model.compile(
        optimizer=Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Training...")
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    # Save plots to Drive
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/mango/accuracy.png")
    plt.close()

    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/mango/loss.png")
    plt.close()


train()

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Loading dataset...
Found 2621 validated image filenames belonging to 8 classes.
Found 656 validated image filenames belonging to 8 classes.
Classes: {'Anthracnose': 0, 'Bacterial Canker': 1, 'Cutting Weevil': 2, 'Die Back': 3, 'Gall Midge': 4, 'Healthy': 5, 'Powdery Mildew': 6, 'Sooty Mould': 7}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 938s 11s/step - accuracy: 0.4234 - loss: 1.6849 - val_accuracy: 0.8750 - val_loss: 0.4943
Epoch 2/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 54s 656ms/step - accuracy: 0.8534 - loss: 0.4991 - val_accuracy: 0.9558 - val_loss: 0.2040
Epoch 3/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 54s 657ms/step - accuracy: 0.9356 - loss: 0.2399 - val_accuracy: 0.9680 - val_loss: 0.1241
Epoch 4/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 55s 668ms/step - accuracy: 0.9520 - loss: 0.1697 - val_accuracy: 0.9802 - val_loss: 0.0730
Epoch 5/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 56s 677ms/step - accuracy: 0.9587 - loss: 0.1338 - val_accuracy: 0.9787 - val_loss: 0.0609
Epoch 6/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 55s 669ms/step - accuracy: 0.9711 - loss: 0.0980 - val_accuracy: 0.9802 - val_loss: 0.0529
Epoch 7/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 81s 657ms/step - accuracy: 0.9724 - loss: 0.0907 - val_accuracy: 0.9878 - val_loss: 0.0396
Epoch 8/8
82/82 ━━━━━━━━━━━━━━━━━━━━ 53s 650ms/step - accuracy: 0.9841 - loss: 0.0675 - val_accuracy: 0.9